# ML 101 · Class 1: Lines That Learn

Linear regression, logistic regression and the perceptron on the **Iris** dataset (Fisher, 1936).

**Task:** binary classification, *setosa* (y = 0) vs *versicolor* (y = 1), using **sepal length** and **sepal width** (cm).

Run the cells from top to bottom (`Shift + Enter`). Each animation cell shows a player under it: press ▶ to watch the decision boundary converge, or use the step buttons to move one frame at a time.

Notation follows the CS229 notes (`main_notes.pdf`, chapters 1 and 2):

| symbol | meaning |
|---|---|
| $x^{(i)}$ | features of flower $i$, with $x_0 = 1$ for the intercept |
| $y^{(i)}$ | label, 0 or 1 |
| $m$ | number of training examples (100) |
| $h_\theta(x)$ | the model's output |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from ml101.common import (Frame, INITIAL_THETA, CLASS_COLORS, CLASS_MARKERS, accuracy,
                          animate_training, boundary_segment, load_binary_iris, pretty,
                          print_summary)

%matplotlib inline

## 0 · The data

We keep two species and two features. Features are **standardized** ($z_j = (x_j - \mu_j)/\sigma_j$) so gradient descent converges quickly; the plots still show centimetres.

To try another problem, change the two lines below, for example `CLASSES = ("versicolor", "virginica")` (overlapping species) or `FEATURES = ("petal_length", "petal_width")`, and re-run the notebook.

In [ ]:
FEATURES = ("sepal_length", "sepal_width")
CLASSES = ("setosa", "versicolor")          # label 0, label 1

data = load_binary_iris(FEATURES, CLASSES)
X, y = data.X, data.y                       # X has the intercept column x0 = 1
print("X shape:", X.shape, "  y counts:", np.bincount(y.astype(int)))
print("first rows of X (standardized, with x0 = 1):\n", X[:3].round(2))

fig, ax = plt.subplots(figsize=(6.5, 4.8))
for label in (0, 1):
    pts = data.X_raw[y == label]
    ax.scatter(pts[:, 0], pts[:, 1], c=CLASS_COLORS[label], marker=CLASS_MARKERS[label],
               edgecolor="white", s=46, label=f"{CLASSES[label]} (y={label})")
ax.set_xlabel(pretty(FEATURES[0])); ax.set_ylabel(pretty(FEATURES[1]))
ax.set_title("Can you draw a line that separates them?")
ax.legend(); plt.show()

## 1 · Linear regression

Model: $\;h_\theta(x) = \theta^T x$

Cost (least squares): $\;J(\theta) = \dfrac{1}{2m} \sum_{i=1}^{m} \big(h_\theta(x^{(i)}) - y^{(i)}\big)^2$

Gradient descent (the LMS rule): $\;\theta := \theta - \alpha \, \dfrac{1}{m} \sum_{i=1}^{m} \big(h_\theta(x^{(i)}) - y^{(i)}\big)\, x^{(i)}$

To classify we predict versicolor when $h_\theta(x) \ge 0.5$, so the **decision boundary** is the line $\theta^T x = 0.5$.
There is also a closed form, the **normal equation** $\theta = (X^TX)^{-1}X^Ty$; it is drawn dashed so you can see gradient descent arrive at it.

In [ ]:
def linreg_h(theta, X):
    return X @ theta

def linreg_cost(theta, X, y):
    return 0.5 * np.mean((linreg_h(theta, X) - y) ** 2)

def linreg_gradient_descent(X, y, lr, iters):
    theta = INITIAL_THETA.copy()                 # a deliberately bad starting line
    history = [Frame(0, theta.copy(), linreg_cost(theta, X, y))]
    for step in range(1, iters + 1):
        error = linreg_h(theta, X) - y           # (m,)
        gradient = X.T @ error / len(y)          # (3,)
        theta -= lr * gradient
        history.append(Frame(step, theta.copy(), linreg_cost(theta, X, y)))
    return theta, history

def normal_equation(X, y):
    return np.linalg.solve(X.T @ X, X.T @ y)

In [ ]:
LR, ITERS = 0.1, 150        # try 0.01 (slow), 1.5 (zig-zag), 1.8 (diverges)

theta_lin, hist_lin = linreg_gradient_descent(X, y, LR, ITERS)
theta_exact = normal_equation(X, y)

print_summary("Linear regression", theta_lin, data, (linreg_h(theta_lin, X) >= 0.5).astype(float))
print("  normal equation theta:", theta_exact.round(3))

In [ ]:
animate_training(data, hist_lin, predict_output=linreg_h, threshold=0.5,
                 title=f"Linear regression: gradient descent, alpha = {LR}",
                 cost_label="J(theta)  (mean squared error / 2)",
                 extra_line=theta_exact, extra_label="normal-equation solution")

**Why it is a rough classifier:** $h_\theta(x)$ is unbounded (it can output 1.7 or −0.4), and least squares penalises points that are correctly classified but far from the line. Far points drag the boundary, so one setosa ends up on the wrong side (99%).

## 2 · Logistic regression

Squash the linear score through the **sigmoid** so the output is a probability:

$$h_\theta(x) = g(\theta^T x), \qquad g(z) = \frac{1}{1+e^{-z}}, \qquad h_\theta(x) = P(y=1 \mid x;\theta)$$

Cost (cross-entropy, from maximum likelihood):

$$J(\theta) = -\frac{1}{m}\sum_{i=1}^{m}\Big[y^{(i)}\log h_\theta(x^{(i)}) + (1-y^{(i)})\log\big(1-h_\theta(x^{(i)})\big)\Big]$$

Its gradient gives **the same update as the LMS rule**: $\;\theta := \theta - \alpha\,\frac{1}{m}\sum_i \big(h_\theta(x^{(i)}) - y^{(i)}\big)x^{(i)}$. Only $h$ changed.

The boundary is where $h = 0.5$, i.e. $\theta^T x = 0$.

In [ ]:
z = np.linspace(-6, 6, 200)
plt.figure(figsize=(5, 2.6))
plt.plot(z, 1 / (1 + np.exp(-z)), color="black", lw=2)
plt.axhline(0.5, color="gray", lw=0.8, ls="--"); plt.axvline(0, color="gray", lw=0.8)
plt.title("sigmoid g(z)"); plt.xlabel("z = θᵀx"); plt.show()

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def logreg_h(theta, X):
    return sigmoid(X @ theta)

def logreg_cost(theta, X, y, eps=1e-12):
    h = np.clip(logreg_h(theta, X), eps, 1 - eps)
    return -np.mean(y * np.log(h) + (1 - y) * np.log(1 - h))

def logreg_gradient_descent(X, y, lr, iters):
    theta = INITIAL_THETA.copy()
    history = [Frame(0, theta.copy(), logreg_cost(theta, X, y))]
    for step in range(1, iters + 1):
        error = logreg_h(theta, X) - y           # same form as linear regression...
        gradient = X.T @ error / len(y)          # ...but h is now the sigmoid
        theta -= lr * gradient
        history.append(Frame(step, theta.copy(), logreg_cost(theta, X, y)))
    return theta, history

In [ ]:
LR, ITERS = 0.5, 600

theta_log, hist_log = logreg_gradient_descent(X, y, LR, ITERS)
print_summary("Logistic regression", theta_log, data, (logreg_h(theta_log, X) >= 0.5).astype(float))
print(f"  final cross-entropy: {hist_log[-1].cost:.4f}")

In [ ]:
animate_training(data, hist_log, predict_output=logreg_h, threshold=0.0,
                 title=f"Logistic regression: gradient descent, alpha = {LR}",
                 cost_label="J(theta)  (cross-entropy)")

**Notice:** the background is the predicted probability. These two species are perfectly separable, so the cost keeps creeping toward 0 and $\theta$ keeps growing (the model gets ever more confident), while the boundary stops moving early. With `CLASSES = ("versicolor", "virginica")` the classes overlap and the cost levels off.

## 3 · The perceptron (Rosenblatt, 1958)

Replace the sigmoid by a hard threshold:

$$h_\theta(x) = g(\theta^T x), \qquad g(z) = \begin{cases}1 & z \ge 0\\ 0 & z < 0\end{cases}$$

and learn **one example at a time**:

$$\theta := \theta + \alpha\,\big(y^{(i)} - h_\theta(x^{(i)})\big)\,x^{(i)}$$

A correct prediction gives $y - h = 0$: no change. A mistake moves the line toward fixing that one flower.
If a separating line exists, the perceptron is guaranteed to find one after finitely many mistakes; if not, it never settles.

In the animation, each frame is one update; the black ring marks the flower that caused it, and the right panel counts misclassified flowers.

In [ ]:
def perceptron_h(theta, X):
    return (X @ theta >= 0).astype(float)

def mistakes(theta, X, y):
    return int(np.sum(perceptron_h(theta, X) != y))

def perceptron_train(X, y, lr, max_epochs, seed=0):
    rng = np.random.default_rng(seed)
    theta = INITIAL_THETA.copy()
    history = [Frame(0, theta.copy(), mistakes(theta, X, y))]
    updates = 0
    for epoch in range(1, max_epochs + 1):
        errors = 0
        for i in rng.permutation(len(y)):        # visit examples in random order
            h = perceptron_h(theta, X[i:i + 1])[0]
            if h != y[i]:                        # only mistakes update theta
                theta += lr * (y[i] - h) * X[i]
                updates += 1
                errors += 1
                history.append(Frame(updates, theta.copy(), mistakes(theta, X, y), highlight=i))
        if errors == 0:
            print(f"converged: a full pass with no mistakes after {epoch} epochs, {updates} updates")
            break
    else:
        print(f"stopped after {max_epochs} epochs: no mistake-free pass")
    return theta, history

In [ ]:
LR, MAX_EPOCHS = 0.1, 100

theta_per, hist_per = perceptron_train(X, y, LR, MAX_EPOCHS)
print_summary("Perceptron", theta_per, data, perceptron_h(theta_per, X))

In [ ]:
# every update is a frame (thinned to 300 frames if the data is not separable)
frames = np.unique(np.linspace(0, len(hist_per) - 1, min(len(hist_per), 300)).astype(int))
animate_training(data, hist_per, predict_output=perceptron_h, threshold=0.0,
                 title=f"Perceptron: one update per mistake, alpha = {LR}",
                 cost_label="misclassified training examples",
                 frame_ids=frames, fps=4)

## 4 · Side by side

All three compute the same score $z = \theta^T x$ and use the same update shape $\theta := \theta + \alpha\,(y - h_\theta(x))\,x$. What they do with $z$ is the whole difference:

| | linear regression | logistic regression | perceptron |
|---|---|---|---|
| $h_\theta(x)$ | $z$ | $1/(1+e^{-z})$ | $1$ if $z \ge 0$ else $0$ |
| cost | squared error | cross-entropy | mistakes only |
| boundary | $\theta^Tx = 0.5$ | $\theta^Tx = 0$ | $\theta^Tx = 0$ |

In [ ]:
models = [
    ("linear regression",   theta_lin, 0.5, (linreg_h(theta_lin, X) >= 0.5), "-",  "#1b1f2a"),
    ("logistic regression", theta_log, 0.0, (logreg_h(theta_log, X) >= 0.5), "--", "#a3245b"),
    ("perceptron",          theta_per, 0.0, perceptron_h(theta_per, X),       ":",  "#1f7a6d"),
]

pad = 0.4
xlim = (data.X_raw[:, 0].min() - pad, data.X_raw[:, 0].max() + pad)
ylim = (data.X_raw[:, 1].min() - pad, data.X_raw[:, 1].max() + pad)
fig, ax = plt.subplots(figsize=(8, 6))
for label in (0, 1):
    pts = data.X_raw[y == label]
    ax.scatter(pts[:, 0], pts[:, 1], c=CLASS_COLORS[label], marker=CLASS_MARKERS[label],
               edgecolor="white", s=46, label=f"{CLASSES[label]} (y={label})")
for name, theta, threshold, pred, style, color in models:
    acc = accuracy(pred.astype(float), y)
    print(f"{name:<20} theta = {np.array2string(theta, precision=2):<24} accuracy = {acc:.0%}")
    ax.plot(*boundary_segment(theta, threshold, data, xlim, ylim), style, color=color, lw=2.2,
            label=f"{name} ({acc:.0%})")
ax.set_xlim(*xlim); ax.set_ylim(*ylim)
ax.set_xlabel(pretty(FEATURES[0])); ax.set_ylabel(pretty(FEATURES[1]))
ax.set_title("Three linear classifiers, three different boundaries")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False)
plt.tight_layout(); plt.show()

## Exercises

1. Find the largest learning rate for which linear regression still converges. Compare it with $2/\lambda_{\max}$, where $\lambda_{\max}$ is the largest eigenvalue of $X^TX/m$ (`np.linalg.eigvalsh(X.T @ X / len(X)).max()`).
2. Remove the standardization (use `add_intercept(data.X_raw)` instead of `data.X`). How many more iterations does gradient descent need? Why?
3. Set `CLASSES = ("versicolor", "virginica")` and re-run. Does the perceptron converge? What happens to logistic regression's cost?
4. Use `FEATURES = ("petal_length", "petal_width")`. Which species pairs become separable?
5. Add an outlier: a versicolor with sepal length 9 cm. Which boundary moves the most? Explain using the cost functions.

To save any animation as a GIF for slides, add `save="figures/name.gif"` to the `animate_training(...)` call.